#Gold Weather Impact Summary

This notebook creates the Gold table:

`gold_weather_impact_summary`

Business goal:
- summarize daily weather conditions by region
- identify weather-related operational risks
- create analytics-ready KPIs for reporting and dashboards

Aggregation grain:
- report_day
- region

In [0]:
import yaml
from pyspark.sql import functions as F

# Load project config
config_path = "/Workspace/Repos/Mini Projects/Vattenfall-Engineering-Capstone-Project/config/project_config.yml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

# Catalog and schemas
catalog = config["catalog"]              # e.g., vattenfall_dev
silver_schema = "refined"
gold_schema = "gold"                     # target Gold schema

# Fully qualified table names
silver_weather = f"{catalog}.{silver_schema}.silver_weather"
gold_silver_weather = f"{catalog}.{gold_schema}.gold_silver_weather"

In [0]:

print(f"Reading source table: {silver_weather}")

weather_df = spark.table(silver_weather)

rows_read = weather_df.count()

print(f"Rows read: {rows_read}")

display(weather_df.limit(10))

## Create Daily Weather Aggregation

Aggregate weather conditions by:
- report_day
- region

Business metrics:
- average temperature
- average wind speed
- total precipitation
- weather alert counts

In [0]:
gold_weather_df = (
    weather_df
    .groupBy("report_day", "region")
    .agg(
        F.avg("temperature_c").alias("avg_temperature_c"),

        F.avg("wind_speed_kmh").alias("avg_wind_speed_kmh"),

        F.sum("precipitation_mm").alias("total_precipitation_mm"),

        F.sum(
            F.when(F.col("weather_alert_level") == "HIGH", 1).otherwise(0)
        ).alias("high_alert_count"),

        F.sum(
            F.when(F.col("weather_alert_level") == "MEDIUM", 1).otherwise(0)
        ).alias("medium_alert_count"),

        F.sum(
            F.when(F.col("weather_alert_level") == "LOW", 1).otherwise(0)
        ).alias("low_alert_count")
    )
)

print("Aggregation grain: report_day + region")

print("Business metrics created:")
print("- average temperature")
print("- average wind speed")
print("- total precipitation")
print("- weather alert counts")

## Add Weather Risk Labels

Create a business-friendly weather risk indicator.

Risk logic:
- HIGH_RISK
- MEDIUM_RISK
- NORMAL

This helps identify regions where weather conditions
may affect operations.

In [0]:
gold_weather_df = (
    gold_weather_df
    .withColumn(
        "weather_risk_label",
        F.when(
            (F.col("avg_wind_speed_kmh") > 40) |
            (F.col("high_alert_count") > 0) |
            (F.col("total_precipitation_mm") > 50),
            "HIGH_RISK"
        )
        .when(
            (F.col("avg_wind_speed_kmh") > 25),
            "MEDIUM_RISK"
        )
        .otherwise("NORMAL")
    )
)

display(gold_weather_df)

## Create Gold Schema

Ensure the Gold schema exists before writing the Gold table.

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS vattenfall_dev.gold")

## Define Gold Target Table

Target table:
- `gold_weather_impact_summary`

This Gold table is designed for:
- dashboards
- operational reporting
- weather risk analysis

In [0]:
target_table = "vattenfall_dev.gold.gold_weather_impact_summary"

print(f"Target table: {target_table}")

## Write Gold Table

Write the aggregated weather impact summary
to the Gold layer using Delta format.

In [0]:
(
    gold_weather_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

rows_written = gold_weather_df.count()

print(f"Rows written: {rows_written}")

print("✅ gold_weather_impact_summary created successfully")

## Validate Gold Output

Inspect the final Gold dataset to confirm:
- aggregation quality
- weather metrics
- risk labels
- dashboard readiness

In [0]:
display(
    gold_weather_df.limit(5)
)

## Improve Numeric Readability

Round numeric business metrics to improve:
- dashboard readability
- reporting quality
- presentation clarity

This step makes Gold outputs more business-friendly
and easier to interpret.

In [0]:
gold_weather_df = (
    gold_weather_df
    .withColumn(
        "avg_temperature_c",
        F.round("avg_temperature_c", 2)
    )
    .withColumn(
        "avg_wind_speed_kmh",
        F.round("avg_wind_speed_kmh", 2)
    )
    .withColumn(
        "total_precipitation_mm",
        F.round("total_precipitation_mm", 2)
    )
)

In [0]:
display(
    gold_weather_df.limit(5)
)